# E3 UNE-EN ISO 5801:2019 Fans Performance testing using standardized airways

## Introduction

The most important document in Europe, for the correct design and operation of a test bench of an axial fan is the **UNE-EN ISO 5801 Standard**, with the 2019 version as currently in effect. It is available at the UPC library in [this link](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0061895).

In the [`E1`](E1_measurement_of_magnitudes.ipynb) notebook we learned the fundamental principles for measuring the main magnitudes in an axial fan: flow rate, pressure, temperature, torque, rotational speed.... In the [`E2`](./E2_uncertainty_management.ipynb) we used the `uncertainty` python package to extend the measurements to report the error. In the present notebook, we apply the guides provided by the Standard to specific test configurations, due to the presence of ducts, wall proximity, inlet/outlet disturbance, ...

We will process laboratory data from a standardized **Category A Inlet-Chamber Test Rig** and propagate measurement uncertainties to verify compliance with ISO 5801 Clause 17 quality criteria.

## Learning Objectives

- **Identify** the four standard **Fan Installation Categories** (A, B, C, D) and understand the definitions of **Fan Pressure ($p_f$)**, **Fan Dynamic Pressure ($p_{fd}$)**, and **Fan Static Pressure ($p_{fs}$)**.
- **Process** raw experimental data from an **ISO 5801 Type 2 / Category A Inlet Chamber** using a multi-nozzle flow meter.
- **Apply** the Python `uncertainties` package to calculate fan characteristic curves ($\Phi - \Psi$, $q_V - p_f$, $\eta_f$) complete with **error bars and 95% confidence bands**.
- **Evaluate** whether the experimental test results fulfill the **maximum allowable uncertainty limits** specified in ISO 5801 Table 13.

## Previous tasks

The ISO 5801 Standard is almost 150 pages long... You don't need to read it all, for the moment... Review to following clauses along with `E1` and `E2` notebooks

- [] Installation categories (Clauses 5 and 6)
- [] Fan pressures and power (Clauses 3 and 15)
- [] Flow rate measurement (Clause 12.5 and Appendix A)
- [] Uncertainty limits (Clause 17, tables 12 and 13) (already read in `E2 notebook`)
- [] Humid Air Properties (Clause 12.9.1, 12.9.2 and Annex H)


## Some simple questions

1. In the `E1` notebook flow rate was measured using an in-duct ISA 1932 nozzle. For the category A, an inlet chamber is preferred for testing low-pressure axial fan. What do you think is the reason for that? How does the large cross-sectional area of the chamber ($A_3 > 5 A_1$) simplify the measurement of inlet total pressure ($p_{sg1}$)?
    
    Answer:

    ***

2. If a laboratory measurement yields an uncertainty in the fan performance of $u_\eta = 4\%$, what steps should be taken to reduce it?
   
   Answer:

   ***

3. In which cases should be the compressibility of air be considered for the fan performance measurement? 
   
   Answer:

   ***

4. Water vapor ($H_2O$, molar mass $\approx 18\text{ g/mol}$) is lighter than dry air ($N_2/O_2$ mixture, molar mass $\approx 29\text{ g/mol}$).
   1. At identical ambient temperature ($20^\circ\text{C}$) and barometric pressure ($101325\text{ Pa}$), is moist air at $70\%$ relative humidity **more dense** or **less dense** than completely dry air ($0\%$ humidity)?
   2. What systematic error (overestimation or underestimation) occurs in mass flow rate ($q_m$) and air power ($P_u$) if humidity is ignored and $R_{\text{dry}}$ is used instead of $R_{\text{wet}}$?
   
   Answer:

   ***

## Tasks

We are now switching to the computation of the performance of an axial fan, similar to the one studied in the CFD notebooks. The main characteristics and ambient conditions are:

- Fan Diameter ($D$): $0.315\text{ m}$ ($A_2 = \pi D^2 / 4$).
- Multi-nozzle area (Figure A.2): $A_n = 0.0125\text{ m}^2$, discharge coefficient $\alpha = 0.985$ [27].
- Barometric Pressure: $p_a = 100500 \pm 100\text{ Pa}$.
- Ambient Temperature: $T_a = 20.0 \pm 0.2^\circ\text{C}$.
- Relative Humidity: $h_{\text{rel}} = 55.0 \pm 3.0\%$ ($0.55 \pm 0.03$).

Operating Points Data:
- `dp_nozzle` ($\Delta p$ across nozzles in Pa): $[450 \pm 0.5, 350 \pm 0.5, 250 \pm 0.5, 150 \pm 0.5, 60 \pm 0.5]$
- `p_chamber` ($p_{e3}$ gauge pressure in Pa, negative): $[-280 \pm 0.5, -220 \pm 0.5, -160 \pm 0.5, -100 \pm 0.5, -35 \pm 0.5]$
- `torque` ($T_r$ torque in N·m): $[0.42 \pm 0.005, 0.45 \pm 0.005, 0.46 \pm 0.005, 0.41 \pm 0.005, 0.35 \pm 0.005]$
- `RPM` (rotational speed in RPM): $[2276 \pm 2, 2274 \pm 2, 2275 \pm 2, 2275 \pm 2, 2276 \pm 2]$

We consider that there is no variation on temperature.

### Write a python script that makes the computation and plot the curves

1. First import the needed modules:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uncertainties.unumpy as unp
from uncertainties import ufloat
```
2. Define all the variables
3. Compute the saturation vapor pressure (Eq. (26)), the partial vapor pressure, the gas constant for wet air (Eq. (25)) and, finally, the density of humid air.
4. Compute the mass and volumetric flow rates
5. Compute fan static, dynamic and total pressure.
6. Compute power and efficiency (both static and total)
7. Check if the relative errors are compliant with ISO 5801 Standard limits (Table 13) 
8. Plot total pressure, power and efficiency against volumetric flow rate, with the error bars.
9. Compute and plot the head and flow coefficients (see [`E1`](../1_Design/D1_fundamentals_and_dimensional_analysis.ipynb) notebook), also with the error bars. 

### Alternative computation of density of humid air

The package [`pyfluids`](https://github.com/portyanikhin/pyfluids), that is a python wrapper of [`CoolProp`](https://coolprop.org/), can accurate and easily compute the density of [humid air](https://github.com/portyanikhin/pyfluids#methods-of-humidair-instances). Here follows an example. Compare your computation following IS0 5801 indications with the density computed with `pyfluids`. 

In [ ]:
from pyfluids import HumidAir, InputHumidAir

humid_air = HumidAir().with_state(
    InputHumidAir.temperature(35), # in Celsius degress 
    InputHumidAir.relative_humidity(70), # in % 
    InputHumidAir.pressure(101000) # absolute pressure in Pa
    ) 

print(f"Humid Air Density: {humid_air.density} kg/m³")

Humid Air Density: 1.1253592049303383 kg/m³


## Your project

With respect to your project:
- Specify which installation category (A, B, C or D) describes your project's fan mounting.
- Make a search in internet, articles, book, ... of similar test benches.
- Make an estimation of the dimensions of your test bench:
  - Length and diameter of conducts
  - Dimensions of flowmeter (or flowmeters if there are more than one)
  - Dimensions of chambers, if needed
  - Method of modification of flow rate (operating points)
- Draw a sketch. No need for now to be accurate. You can use [draw.oi](https://app.diagrams.net/) for instance